# URL Uploader Bot — Google Colab

This notebook launches the latest `main` branch of [Telebotfaroff/URL-uploader](https://github.com/Telebotfaroff/URL-uploader).

## Before running
1. In Colab, open **Secrets** (key icon in the left sidebar).
2. Add `BOT_TOKEN`, `API_ID`, `API_HASH`, and `OWNER_ID`.
3. Add `DATABASE_URL` as a Colab Secret using the connection string from your Neon PostgreSQL project. This is needed for persistent channel upload history and the **Skip already uploaded** feature to work across runtime resets.
4. Enable notebook access for each secret.
5. Run every cell from top to bottom.

`OWNER_ID` must be your numeric Telegram user ID. Other optional settings: `LOG_CHANNEL`, `ADMIN`, `BANNED_USERS`, `HTTP_PROXY`, `CHUNK_SIZE`, and `DEF_THUMB_NAIL_VID_S`. Secrets are loaded without printing their values.

After changing code or secrets, stop any old bot instance and rerun this notebook from the beginning.

## Dump channel (optional)

After the bot starts, send `/setdump @yourchannelusername` or `/setdump -1001234567890` to configure the upload destination. The bot must be an administrator of that channel with permission to post. The setting is stored in Neon and persists across Colab runtime resets. Send `/setdump` to check the saved destination, or `/setdump off` to disable it. Uploads go to the dump channel first; if that send fails, the bot retries in your chat. The source-check cell verifies this feature before the bot starts.

## Parallel channel processing

For `/channel` and `/actress` batch downloads, the bot runs a bounded pipeline with up to **2 concurrent downloads and 2 concurrent Telegram uploads**. Videos use unique temporary filenames so parallel downloads cannot overwrite each other. These limits help overlap downloading and uploading without launching an unbounded number of transfers. `/vid` single-video mode remains unchanged.


In [ ]:
# Load Colab Secrets into environment variables without displaying their values.
from google.colab import userdata
import os

REQUIRED_SECRETS = ("BOT_TOKEN", "API_ID", "API_HASH", "OWNER_ID")
OPTIONAL_SECRETS = (
    "DATABASE_URL", "LOG_CHANNEL", "ADMIN", "BANNED_USERS",
    "HTTP_PROXY", "CHUNK_SIZE", "DEF_THUMB_NAIL_VID_S",
)

missing = []
for key in REQUIRED_SECRETS:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value is None or not str(value).strip():
        missing.append(key)
    else:
        os.environ[key] = str(value).strip()

if missing:
    raise RuntimeError(
        "Missing Colab Secrets: " + ", ".join(missing)
        + ". Add them in the Secrets panel and enable notebook access."
    )

for key in OPTIONAL_SECRETS:
    try:
        value = userdata.get(key)
    except Exception:
        value = None
    if value is not None and str(value).strip():
        os.environ[key] = str(value).strip()

try:
    int(os.environ["API_ID"])
    owner_id = int(os.environ["OWNER_ID"])
    if owner_id <= 0:
        raise ValueError
except ValueError as exc:
    raise ValueError("API_ID and OWNER_ID must be numeric; OWNER_ID must be positive.") from exc

database_url = os.environ.get("DATABASE_URL", "").strip()
if database_url and not database_url.startswith(("postgres://", "postgresql://")):
    raise ValueError("DATABASE_URL should be a PostgreSQL connection string from Neon. The secret value was not displayed.")
if database_url:
    print("Neon DATABASE_URL detected; persistent upload history is configured.")
else:
    print("WARNING: DATABASE_URL is not set. The bot can start, but persistent channel upload history / skip checks will not work reliably.")

print("Required credentials loaded and basic configuration validated.")


## 1. Install system dependencies

Install FFmpeg and update pip. Python packages are installed after cloning so the notebook uses the current repository's requirements file.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade pip

## 2. Download the latest repository

This starts from a clean copy of the current `main` branch in the Colab runtime.

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/URL-uploader")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 --branch main https://github.com/Telebotfaroff/URL-uploader.git /content/URL-uploader
%cd /content/URL-uploader

print("Repository ready on branch main.")

## 3. Install project dependencies and check the source

The syntax check catches Python compilation errors before the bot starts.

In [ ]:
%cd /content/URL-uploader
!python -m pip install -q -r requirements.txt
!python -m pip install -q "pyrogram==2.0.106" tgcrypto requests
!python -m compileall -q bot.py plugins

# Verify dump-channel configuration and parallel channel pipeline support.
from pathlib import Path
javdl_source = Path("plugins/javdl.py").read_text(encoding="utf-8")
database_source = Path("plugins/database/database.py").read_text(encoding="utf-8")
downloader_source = Path("plugins/javdl_core/downloader.py").read_text(encoding="utf-8")
if 'command(["setdump"]' not in javdl_source or 'get_setting("dump_chat_id")' not in javdl_source:
    raise RuntimeError("Dump-channel commands are missing from the checked-out source. Confirm the main branch is up to date.")
if "async def get_setting(" not in database_source or "async def set_setting(" not in database_source:
    raise RuntimeError("Persistent bot settings are missing from the database module.")
if "asyncio.Semaphore(2)" not in javdl_source or "upload_semaphore=upload_semaphore" not in javdl_source:
    raise RuntimeError("The two-download/two-upload channel pipeline is missing from the checked-out source.")
if "time.time_ns()" not in downloader_source:
    raise RuntimeError("Unique download filenames are missing; parallel downloads could conflict.")

print("Dependencies installed and Python source compilation passed.")
print("Dump-channel feature verified: /setdump, Neon persistence, and upload fallback.")
print("Parallel channel pipeline verified: up to 2 downloads and 2 uploads at once.")


## 4. Verify Telegram credentials and start the bot

The next cell verifies the bot token with Telegram, then starts `bot.py` with live logs in this output and appends them to `/content/URL-uploader/colab_bot.log`.

The bot is owner-only: send `/start` from the Telegram account whose numeric user ID matches `OWNER_ID`. If Telegram reports that another instance is already running, stop the other copy using the same bot token.

This cell remains active while the bot runs. Stop the cell to stop the bot. Colab is temporary hosting and may disconnect.

In [ ]:
import os
import subprocess
import sys
import requests
from datetime import datetime

token = os.environ["BOT_TOKEN"]
owner_id = int(os.environ["OWNER_ID"])

try:
    response = requests.get(
        f"https://api.telegram.org/bot{token}/getMe",
        timeout=20,
    )
    response.raise_for_status()
    result = response.json()
except requests.RequestException as exc:
    raise RuntimeError(f"Could not verify BOT_TOKEN with Telegram: {exc}") from exc

if not result.get("ok"):
    raise RuntimeError("Telegram rejected BOT_TOKEN. Check the BOT_TOKEN Colab Secret.")

bot_info = result["result"]
print(f"Token verified. Bot: @{bot_info.get('username', 'unknown')}", flush=True)
print(f"Configured owner Telegram user ID: {owner_id}", flush=True)
print("Only the configured owner should use this bot.", flush=True)

os.makedirs("DOWNLOADS", exist_ok=True)
LOG_PATH = "/content/URL-uploader/colab_bot.log"
print(f"\nStarting bot. Live logs will appear below and be saved to: {LOG_PATH}\n" + "-" * 72, flush=True)

process = subprocess.Popen(
    [sys.executable, "-u", "bot.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

try:
    with open(LOG_PATH, "a", encoding="utf-8", buffering=1) as log_file:
        log_file.write(
            f"\n{'=' * 24} Session started "
            f"{datetime.now().isoformat(timespec='seconds')} {'=' * 24}\n"
        )
        for line in process.stdout:
            log_file.write(line)
            print(line, end="", flush=True)
    return_code = process.wait()
    print(f"\nBot process exited with code {return_code}.", flush=True)
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    print("\nBot stopped.", flush=True)
finally:
    if process.stdout:
        process.stdout.close()

print("To inspect saved logs, run: !tail -n 200 /content/URL-uploader/colab_bot.log")